In [1]:
!pip install "langchain==0.3.25" "langchain-community==0.3.24" "langchain-text-splitters==0.3.8" "langchain-google-genai==2.1.5" faiss-cpu sentence-transformers pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 108.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take 

In [2]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

DATA_PATH = "/content/drive/MyDrive/my_personal_data"
os.makedirs(DATA_PATH, exist_ok=True)  # creates the folder if it doesn't exist yet

documents = []
documents += DirectoryLoader(DATA_PATH, glob="**/*.txt", loader_cls=TextLoader).load()
documents += DirectoryLoader(DATA_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader).load()

print(f"Loaded {len(documents)} documents")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 10 documents


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 10 chunks


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("/content/drive/MyDrive/my_rag_index")
print("Vector store saved.")

/tmp/ipykernel_417/458717836.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store saved.


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.3)

prompt_template = """You are a helpful assistant that answers questions using only the context below, which is personal information about the user. If the answer isn't in the context, say you don't know — don't make things up.

Context:
{context}

Question: {question}
Answer:"""

prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

In [7]:
response = qa_chain.invoke({"query": "what is aI"})
print(response["result"])

Based on the provided context, AI stands for Artificial Intelligence. Rather than a replacement for educators, it is described as their "most powerful assistant" and a partner in the classroom designed to augment human intelligence, amplify creativity and empathy, and handle routine tasks such as automated grading and scheduling.


In [8]:
import os

DEPLOY_PATH = "/content/drive/MyDrive/whatsapp-rag-deploy"
os.makedirs(DEPLOY_PATH, exist_ok=True)

# Create requirements.txt
requirements_content = """
langchain==0.3.25
langchain-community==0.3.24
langchain-text-splitters==0.3.8
langchain-google-genai==2.1.5
faiss-cpu
sentence-transformers
pypdf
Flask # Added Flask for a simple web app example
"""

with open(os.path.join(DEPLOY_PATH, "requirements.txt"), "w") as f:
    f.write(requirements_content)

print(f"Created {os.path.join(DEPLOY_PATH, 'requirements.txt')}")

Created /content/drive/MyDrive/whatsapp-rag-deploy/requirements.txt


In [9]:
# Create app.py
app_content = """
import os
from flask import Flask, request, jsonify
from google.colab import userdata
from google.colab import drive

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

app = Flask(__name__)

# Set GOOGLE_API_KEY from Colab secrets (or environment variable if deployed outside Colab)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY', os.environ.get('GOOGLE_API_KEY'))

# Mount Google Drive (necessary for accessing saved index)
drive.mount('/content/drive', force_remount=True)

# Define paths
VECTOR_STORE_PATH = "/content/drive/MyDrive/my_rag_index"

# Initialize embeddings and load vector store
# Note: This is done globally for efficiency, but consider lazy loading or other patterns for production
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(VECTOR_STORE_PATH, embeddings, allow_dangerous_deserialization=True)

# Initialize LLM and QA Chain
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.3)

prompt_template = '''You are a helpful assistant that answers questions using only the context below, which is personal information about the user. If the answer isn't in the context, say you don't know - don't make things up.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:'''

prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

@app.route('/query', methods=['POST'])
def query_rag():
    data = request.get_json()
    user_query = data.get('query')

    if not user_query:
        return jsonify({"error": "Please provide a 'query' in the request body"}), 400

    try:
        response = qa_chain.invoke({"query": user_query})
        return jsonify({"result": response["result"]})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    # For local testing, you might run this with 'python app.py'
    # For deployment, the platform will typically handle how the app is run (e.g., gunicorn)
    app.run(host='0.0.0.0', port=int(os.environ.get('PORT', 8080)))
"""

with open(os.path.join(DEPLOY_PATH, "app.py"), "w") as f:
    f.write(app_content)

print(f"Created {os.path.join(DEPLOY_PATH, 'app.py')}")

Created /content/drive/MyDrive/whatsapp-rag-deploy/app.py


The `requirements.txt` and `app.py` files have been created in the `whatsapp-rag-deploy` directory. You can now proceed with deploying your application using these files.

**To test the `app.py` locally (for demonstration purposes, this requires port forwarding in Colab or running in a different environment):**

1.  **Navigate to the directory:**
    ```bash
    %cd /content/drive/MyDrive/whatsapp-rag-deploy
    ```
2.  **Install dependencies:**
    ```bash
    !pip install -r requirements.txt
    ```
3.  **Run the Flask app:**
    ```bash
    !python app.py
    ```

Once the Flask app is running, you can send POST requests to the `/query` endpoint. For example, using `curl` from another terminal or a tool like Postman:

```bash
curl -X POST -H "Content-Type: application/json" -d '{"query": "what is AI"}' http://localhost:8080/query
```